<div align="center">
<p align="center" style="width: 100%;">
    <img src="https://raw.githubusercontent.com/vlm-run/.github/refs/heads/main/profile/assets/vlm-black.svg" alt="VLM Run Logo" width="80" style="margin-bottom: -5px; color: #2e3138; vertical-align: middle; padding-right: 5px;"><br>
</p>
<p align="center"><a href="https://docs.vlm.run"><b>Website</b></a> | <a href="https://docs.vlm.run/"><b>API Docs</b></a> | <a href="https://docs.vlm.run/blog"><b>Blog</b></a> | <a href="https://discord.gg/AMApC2UzVY"><b>Discord</b></a>
</p>
<p align="center">
<a href="https://discord.gg/AMApC2UzVY"><img alt="Discord" src="https://img.shields.io/badge/discord-chat-purple?color=%235765F2&label=discord&logo=discord"></a>
<a href="https://twitter.com/vlmrun"><img alt="Twitter Follow" src="https://img.shields.io/twitter/follow/vlmrun.svg?style=social&logo=twitter"></a>
</p>
</div>

Welcome to **[VLM Run Cookbooks](https://github.com/vlm-run/vlmrun-cookbook)**, a comprehensive collection of examples and notebooks demonstrating the power of structured visual understanding using the [VLM Run Platform](https://app.vlm.run).


## Image Captioning & Tagging with VLM Run (Node.js)

This notebook demonstrates how to use the **VLM Run Node.js SDK** to generate detailed captions and tags for images using **Node.js/TypeScript**. Image captioning is perfect for:
- **Accessibility**: Creating alt-text descriptions for images
- **Content Management**: Automatically organizing and cataloging image libraries
- **SEO**: Generating image descriptions for better search engine optimization
- **Automated Analysis**: Building workflows that understand visual content

We'll use VLM Run's vision-language models through the native SDK to generate comprehensive, contextual captions and extract relevant tags from various images.


### Environment Setup

To get started, install the required packages and sign up for an API key on the [VLM Run App](https://app.vlm.run).
- Store the VLM Run API key under the `VLMRUN_API_KEY` environment variable.


## Prerequisites

* Node.js 18+
* VLM Run API key (get one at [app.vlm.run](https://app.vlm.run))
* Deno or tslab kernel for running TypeScript in Jupyter


## Setup

First, let's install the required packages:


In [1]:
// Install the VLM Run SDK
// npm install vlmrun zod zod-to-json-schema

// If using Deno kernel, install dependencies via npm specifiers
// For tslab, run: npm install vlmrun zod zod-to-json-schema in your project directory


In [2]:
// Import the VLM Run SDK and dependencies
import { VlmRun } from "npm:vlmrun";
import { z } from "npm:zod";
import { zodToJsonSchema } from "npm:zod-to-json-schema";


In [3]:
// Get API key from environment variable
const VLMRUN_API_KEY = Deno.env.get("VLMRUN_API_KEY");

if (!VLMRUN_API_KEY) {
    throw new Error("Please set the VLMRUN_API_KEY environment variable");
}

console.log("✓ API Key loaded successfully");


✓ API Key loaded successfully


Let's initialize the VLM Run client using the SDK


In [ ]:
// Initialize the VLM Run client using the SDK
const client = new VlmRun({
    apiKey: VLMRUN_API_KEY,
    baseURL: "https://agent.vlm.run/v1"  // Use the agent API endpoint
});

console.log("✓ VLM Run SDK client initialized!");


✓ VLM Run SDK client initialized!


## Example 1: Simple Image Captioning

Let's start with a simple example - generating a caption for a single image using the VLM Run SDK.


In [8]:
// Example image URL
const imageUrl = "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.object-detection/crossroad.jpg";

console.log("📷 Image URL:", imageUrl);
console.log("\n🔍 Generating caption...\n");

// Use the agent.completions interface (OpenAI-compatible)
const response = await client.agent.completions.create({
    model: "vlmrun-orion-1:auto",
    messages: [
        {
            role: "user",
            content: [
                { type: "text", text: "Generate a detailed caption for this image" },
                { type: "image_url", image_url: { url: imageUrl, detail: "auto" } }
            ]
        }
    ]
});

const caption = response.choices[0].message.content;
console.log(`Caption: ${caption}`);


📷 Image URL: https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.object-detection/crossroad.jpg

🔍 Generating caption...

Caption: A bustling metropolitan scene unfolds on a busy street, likely in New York City, framed by towering multi-story buildings that create a classic urban canyon. The architecture is a mix of ornate historical masonry with arched windows and more modern, streamlined structures, with some scaffolding visible on the right. In the foreground, a vibrant yellow Toyota RAV4 taxi with the medallion number 1P55 is stopped at a crosswalk, where several pedestrians are active: a man in a light blue shirt and tan pants walks while using his smartphone, a woman in a dark jacket pushes a stroller accompanied by a young child in a beige coat, and another man with a black backpack stands nearby. The street is filled with vehicles, including a dark red SUV, a white commercial van, and a silver SUV receding into the distance. Notable signs include a pink neon 

## Example 2: Helper Function

Let's create a reusable helper function for captioning:


In [9]:
/**
 * Generate a caption for an image using VLM Run SDK.
 * 
 * @param imageUrl - URL of the image to caption
 * @param prompt - Custom prompt for captioning (optional)
 * @returns Generated caption string
 */
async function generateCaption(
    imageUrl: string,
    prompt: string = "Generate a detailed caption for this image"
): Promise<string> {
    const response = await client.agent.completions.create({
        model: "vlmrun-orion-1:auto",
        messages: [
            {
                role: "user",
                content: [
                    { type: "text", text: prompt },
                    { type: "image_url", image_url: { url: imageUrl, detail: "auto" } }
                ]
            }
        ]
    });
    
    return response.choices[0].message.content || "";
}

console.log("✓ Helper function defined");


✓ Helper function defined


## Example 3: Structured Output with Zod Schema

For production applications, you'll want structured JSON output. Let's use Zod schemas with the VLM Run SDK:


In [10]:
// Define the schema using Zod
const ImageCaptionSchema = z.object({
    caption: z.string().describe("Detailed caption of the scene"),
    tags: z.array(z.string()).describe("Tags that describe the image"),
    primary_objects: z.array(z.string()).describe("Main objects visible in the image"),
    scene_type: z.string().describe("Type of scene (e.g., indoor, outdoor, street, nature)"),
    mood: z.string().optional().describe("Overall mood or atmosphere of the image")
});

// Type inference from schema
type ImageCaption = z.infer<typeof ImageCaptionSchema>;

console.log("✓ ImageCaption schema defined");


✓ ImageCaption schema defined


In [12]:
/**
 * Generate a structured caption using VLM Run SDK with Zod schema validation.
 * 
 * @param imageUrl - URL of the image to analyze
 * @returns Validated ImageCaption object
 */
async function generateStructuredCaption(imageUrl: string): Promise<ImageCaption> {
    const response = await client.agent.completions.create({
        model: "vlmrun-orion-1:auto",
        messages: [
            {
                role: "user",
                content: [
                    { 
                        type: "text", 
                        text: "Analyze this image and provide a detailed caption, relevant tags, primary objects, scene type, and mood." 
                    },
                    { 
                        type: "image_url", 
                        image_url: { url: imageUrl, detail: "auto" } 
                    }
                ]
            }
        ],
        response_format: { 
            type: "json_schema", 
            schema: zodToJsonSchema(ImageCaptionSchema)
        } as any
    });
    
    const rawContent = response.choices[0].message.content || "{}";
    
    // Parse and validate the response with Zod
    const parsed = JSON.parse(rawContent);
    return ImageCaptionSchema.parse(parsed);
}

// Generate structured caption
console.log("🔍 Generating structured caption...\n");
const structuredResult = await generateStructuredCaption(imageUrl);

console.log("✓ Validated Object:");
console.log(structuredResult);
console.log("\n📝 Formatted Output:");
console.log(`Caption: ${structuredResult.caption}`);
console.log(`Tags: ${structuredResult.tags.join(", ")}`);
console.log(`Primary Objects: ${structuredResult.primary_objects.join(", ")}`);
console.log(`Scene Type: ${structuredResult.scene_type}`);
if (structuredResult.mood) {
    console.log(`Mood: ${structuredResult.mood}`);
}


🔍 Generating structured caption...

✓ Validated Object:
{
  caption: 'A vibrant and bustling street scene in a dense urban environment, likely Midtown Manhattan near West 36th Street. A family, including a man, a woman pushing a stroller with a child, and another young child, crosses a zebra crosswalk in the foreground. They are surrounded by typical city traffic, most notably a bright yellow Toyota RAV4 taxi and a dark red SUV. Tall commercial buildings with varied architectural styles, including one with a "Palm" restaurant neon sign and another with "Office" signage, line the street, leading toward distant digital billboards. The scene captures the energetic and dynamic atmosphere of a busy metropolitan day.',
  tags: [
    "urban",       "street scene",
    "city life",   "New York City",
    "yellow taxi", "pedestrians",
    "crosswalk",   "skyscraper",
    "traffic",     "bustle",
    "family",      "Manhattan"
  ],
  primary_objects: [
    "yellow taxi (Toyota RAV4)",
    "pedes

## Example 4: Batch Processing Multiple Images

Process multiple images efficiently:


In [13]:
// Sample images to process
const sampleImages = [
    { url: "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.object-detection/crossroad.jpg", name: "Street Scene" },
    { url: "https://storage.googleapis.com/vlm-data-public-prod/hub/examples/image.caption/car.jpg", name: "Vintage Car" },
    { url: "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=400", name: "Mountain" },
    { url: "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=400", name: "Cat" }
];

interface BatchCaptionResult {
    imageName: string;
    caption: string;
    tags: string;
    sceneType: string;
    error?: string;
}

async function processImagesBatch(images: { url: string; name: string }[]): Promise<BatchCaptionResult[]> {
    const results: BatchCaptionResult[] = [];
    
    for (const img of images) {
        try {
            console.log(`Processing: ${img.name}...`);
            const result = await generateStructuredCaption(img.url);
            results.push({
                imageName: img.name,
                caption: result.caption,
                tags: result.tags.join(", "),
                sceneType: result.scene_type
            });
        } catch (error) {
            console.log(`Error processing ${img.name}: ${error}`);
            results.push({
                imageName: img.name,
                caption: `Error: ${error}`,
                tags: "",
                sceneType: "",
                error: String(error)
            });
        }
    }
    
    return results;
}

// Process all images
const batchResults = await processImagesBatch(sampleImages);

// Display results
console.log("\n=== Batch Processing Results ===\n");
console.table(batchResults.map(r => ({
    Image: r.imageName,
    Caption: r.caption.substring(0, 50) + "...",
    Tags: r.tags,
    Scene: r.sceneType
})));


Processing: Street Scene...
Processing: Vintage Car...
Processing: Mountain...
Processing: Cat...

=== Batch Processing Results ===

┌───────┬────────────────┬─────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────┐
│ (idx) │ Image          │ Caption                                                 │ Tags                                                                                                                                                                              │ Scene                            │
├───────┼────────────────┼─────────────────────────────────────────────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼───

## Example 5: Custom Caption Styles

Customize caption output by adjusting the prompt:


In [15]:
// Different caption styles
const captionStyles: Record<string, string> = {
    "Short & concise": "Generate a brief, one-sentence caption for this image.",
    "Detailed & descriptive": "Generate a detailed, descriptive caption for this image with at least 50 words.",
    "Creative & poetic": "Generate a creative and poetic caption for this image.",
    "SEO-optimized": "Generate an SEO-optimized caption for this image that includes relevant keywords.",
    "Accessibility focused": "Generate an accessibility-focused alt text for this image that would help visually impaired users understand the content."
};

console.log("Different Caption Styles for the Same Image:\n");
console.log("=".repeat(80));

for (const [styleName, prompt] of Object.entries(captionStyles)) {
    const captionResult = await generateCaption(imageUrl, prompt);
    console.log(`\n${styleName.toUpperCase()}:`);
    console.log(captionResult);
    console.log("-".repeat(80));
}


Different Caption Styles for the Same Image:


SHORT & CONCISE:
A busy city street features pedestrians crossing a crosswalk alongside multiple vehicles, with tall buildings lining both sides of the road under a bright sky.
--------------------------------------------------------------------------------

DETAILED & DESCRIPTIVE:
This image captures the bustling energy of a New York City street on a bright day. In the foreground, a man in a light blue shirt and khaki pants walks purposefully across a crosswalk while wearing earbuds, followed by a woman pushing a black stroller and a young child. They are joined by other pedestrians navigating the intersection under a clear blue sky.

The street is filled with active urban traffic, most notably a bright yellow Toyota RAV4 taxi with the medallion "1P55" and a dark red Hyundai SUV. The scene is framed by an "urban canyon" of towering buildings, featuring a mix of classic masonry facades and modern architecture with visible construction scaf

💡 The VLM Run SDK provides a convenient wrapper around the OpenAI client.
   Install it with: npm install vlmrun


## Best Practices & Tips

### Caption Quality
- Be specific in your prompts about the level of detail you need
- For accessibility, ask for alt-text specifically
- For SEO, ask for keyword-rich descriptions

### Performance
- Use batch processing for multiple images
- Consider caching captions to avoid reprocessing
- Use appropriate image sizes (doesn't need to be full resolution)

### TypeScript Benefits
- Use Zod schemas for type-safe structured outputs
- Leverage TypeScript's type inference for better IDE support
- Use async/await for cleaner asynchronous code

### Tag Categories
VLM Run supports various tag types:
- **Objects**: person, car, truck, bus, bicycle, motorcycle
- **Scenes**: street, building, park, forest, beach
- **Time**: morning, afternoon, evening, night
- **Weather**: sunny, cloudy, rainy, snowing


## Additional Resources

- [VLM Run Documentation](https://docs.vlm.run)
- [VLM Run Node.js SDK](https://github.com/vlm-run/vlmrun-node-sdk)
- [API Reference](https://docs.vlm.run/agents/capabilities/image/captioning)
- [More Examples](https://github.com/vlm-run/vlmrun-cookbook)
- [Discord Community](https://discord.gg/AMApC2UzVY)

### Installation

```bash
npm install vlmrun zod zod-to-json-schema
```

### Quick Start

```typescript
import { VlmRun } from "vlmrun";

const client = new VlmRun({
    apiKey: process.env.VLMRUN_API_KEY,
    baseURL: "https://agent.vlm.run/v1"
});

const response = await client.agent.completions.create({
    model: "vlmrun-orion-1:auto",
    messages: [{
        role: "user",
        content: [
            { type: "text", text: "Describe this image" },
            { type: "image_url", image_url: { url: "https://example.com/image.jpg" } }
        ]
    }]
});

console.log(response.choices[0].message.content);
```
